In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

current_dir = Path.cwd()
project_root = current_dir.parent

print("Project Root:", project_root)

Project Root: c:\Users\sneha\OneDrive\Documents\CLV_Project


In [2]:
transactions = pd.read_csv(
    f"{project_root}/Data/transactions_cleaned.csv",
    parse_dates=["order_date"]
)

transactions.head()

,order_id,product_id,product,quantity,order_date,unit_price,customer_id,country,order_value
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [3]:
snapshot_date = transactions["order_date"].max()
print("Snapshot date:", snapshot_date)

Snapshot date: 2011-12-09 12:50:00


In [4]:
customer_features = transactions.groupby("customer_id").agg({
    "order_id": "nunique",
    "order_value": ["sum", "mean", "std"],
    "order_date": ["min", "max"],
    "country": "first"
})

customer_features.columns = [
    "num_orders",
    "total_spent",
    "avg_order_value",
    "order_value_std",
    "first_order_date",
    "last_order_date",
    "country"
]

customer_features = customer_features.reset_index()

customer_features.head()

,customer_id,num_orders,total_spent,avg_order_value,order_value_std,first_order_date,last_order_date,country
0,12346.0,1,77183.60,77183.600000,NaN,2011-01-18 10:01:00,2011-01-18 10:01:00,United Kingdom
1,12347.0,7,4310.00,23.681319,23.289902,2010-12-07 14:57:00,2011-12-07 15:52:00,Iceland
2,12348.0,4,1797.24,57.975484,48.514857,2010-12-16 19:09:00,2011-09-25 13:13:00,Finland
3,12349.0,1,1757.55,24.076027,34.655913,2011-11-21 09:51:00,2011-11-21 09:51:00,Italy
4,12350.0,1,334.40,19.670588,7.275538,2011-02-02 16:01:00,2011-02-02 16:01:00,Norway


In [5]:
customer_features["recency_days"] = (
    snapshot_date - customer_features["last_order_date"]
).dt.days

customer_features["customer_age_days"] = (
    customer_features["last_order_date"]
    - customer_features["first_order_date"]
).dt.days

customer_features["frequency"] = customer_features["num_orders"]
customer_features["monetary"] = customer_features["total_spent"]

In [6]:
transactions["year_month"] = (
    transactions["order_date"].dt.to_period("M")
)

months_active = (
    transactions.groupby("customer_id")["year_month"]
    .nunique()
    .reset_index(name="months_active")
)

customer_features = customer_features.merge(
    months_active,
    on="customer_id",
    how="left"
)

In [7]:
monthly_spend = (
    transactions.groupby(["customer_id", "year_month"])["order_value"]
    .sum()
    .reset_index()
)

growth = monthly_spend.groupby("customer_id")["order_value"].agg(
    first="first",
    last="last"
).reset_index()

growth["purchase_growth"] = (
    (growth["last"] - growth["first"]) /
    (growth["first"] + 1)
)

customer_features = customer_features.merge(
    growth[["customer_id", "purchase_growth"]],
    on="customer_id",
    how="left"
)

In [8]:
customer_features.describe()
customer_features.isna().sum()

customer_id           0
num_orders            0
total_spent           0
avg_order_value       0
order_value_std      71
first_order_date      0
last_order_date       0
country               0
recency_days          0
customer_age_days     0
frequency             0
monetary              0
months_active         0
purchase_growth       0
dtype: int64

In [9]:
output_path = f"{project_root}/Data/customer_features.csv"

customer_features.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: c:\Users\sneha\OneDrive\Documents\CLV_Project/Data/customer_features.csv
